[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/01_ahp_iot_palmor.ipynb)

# Proceso Analítico Jerárquico (AHP) Didáctico
**Toma de Decisiones Multicriterio (MCDA) · Universidad del Magdalena**  
**Docente:** Miguel Ángel Polo-Castañeda  
**Caso de Estudio:** Selección de tecnología de comunicación IoT/WSN (**LoRaWAN, GSM/GPRS, Sigfox, Zigbee**) para monitoreo agroclimático en Palmor (Sierra Nevada de Santa Marta).

---

### 💡 Guía Pedagógica: ¿Cómo funciona AHP sin enredarse en matrices?
En el método AHP (Saaty, 1980), los decisores comparan elementos **por pares** en lugar de asignar notas arbitrarias.
Para no cometer errores de digitación en Python:
1. **Reciprocidad automática:** Si el criterio $A$ es 3 veces más importante que $B$ ($a_{ij} = 3$), automáticamente $B$ respecto a $A$ es su inverso ($a_{ji} = 1/3$).
2. **Diagonal unitaria:** Un elemento comparado consigo mismo siempre vale $1$ ($a_{ii} = 1$).
3. **Escala fundamental de Saaty (1 a 9):**
   * **1:** Igual importancia.
   * **3:** Moderada importancia de uno sobre otro.
   * **5:** Fuerte importancia.
   * **7:** Muy fuerte importancia.
   * **9:** Extrema importancia.
   * *(Valores 2, 4, 6, 8 son intermedios).*

A continuación dispones de **controles interactivos (sliders)** para diligenciar los juicios de forma intuitiva, seguidos del **desglose matemático paso a paso en NumPy** para que entiendas exactamente qué ocurre detrás de cada fórmula.

In [1]:
# 1. Instalación de dependencias (necesario en Google Colab)
!pip install -q pyDecision ipywidgets matplotlib pandas numpy

# 2. Descarga automática del módulo auxiliar didáctico si estás en Google Colab
import os, urllib.request
if not os.path.exists("mcda_didactico.py"):
    url = "https://raw.githubusercontent.com/miguepoloc/toma-decisiones-mcda/main/mcda_didactico.py"
    try:
        urllib.request.urlretrieve(url, "mcda_didactico.py")
        print("✅ Módulo didáctico 'mcda_didactico.py' cargado correctamente.")
    except Exception as e:
        print("⚠️ No se pudo descargar mcda_didactico.py automáticamente:", e)

# 3. Importación de librerías de trabajo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyDecision.algorithm import ahp_method
import mcda_didactico as md

print("Librerías importadas con éxito.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Librerías importadas con éxito.


## Paso 1: Ponderación de Criterios (Panel de 3 Expertos)

Los 4 criterios seleccionados en la Sesión 1 son:
1. **Alcance:** Distancia efectiva de cobertura en topografía montañosa.
2. **Autonomía:** Duración de la batería en nodos remotos sin red eléctrica.
3. **Infraestructura:** Cobertura de red comercial disponible en Palmor.
4. **Madurez:** Viabilidad técnica y soporte comercial de los proveedores.

Para agregar los juicios de los 3 expertos sin sesgar las escalas, se utiliza la **media geométrica** (Forman & Peniwati, 1998):
$$A_{agregada} = \left(\prod_{k=1}^K M_k\right)^{1/K} = (M_1 \odot M_2 \odot M_3)^{1/3}$$

Utiliza el siguiente widget interactivo para explorar o modificar los juicios de cada experto con sliders. El widget calcula en tiempo real la matriz agregada, los pesos relativos y el Ratio de Consistencia ($CR$).

In [2]:
# Widget interactivo para la Matriz de Criterios
# Puedes mover los sliders de cada experto o presionar 'Cargar Caso Palmor' para ver los datos de clase.
widget_criterios = md.AHPWidget(
    elementos=md.PALMOR_CRITERIOS,
    n_expertos=3,
    titulo="Matriz de Criterios (Comparación por Pares)",
    matrices_iniciales=md.PALMOR_MATRICES_CRITERIOS
)
widget_criterios.mostrar()

### 📐 Desglose Matemático Paso a Paso (Criterios en NumPy)

En lugar de usar una "caja negra", ejecutamos las ecuaciones de Saaty de forma explícita:

1. **Suma por columnas:** $S_j = \sum_{i=1}^n a_{ij}$
2. **Normalización por columnas:** $r_{ij} = \frac{a_{ij}}{S_j}$
3. **Vector de prioridades (pesos $w$):** $w_i = \frac{1}{n} \sum_{j=1}^n r_{ij}$
4. **Vector de consistencia:** $(A \cdot w)_i$
5. **Autovalor principal ($\lambda_{\max}$):** $\lambda_{\max} = \frac{1}{n} \sum_{i=1}^n \frac{(A \cdot w)_i}{w_i}$
6. **Índice de Consistencia ($CI$):** $CI = \frac{\lambda_{\max} - n}{n - 1}$
7. **Ratio de Consistencia ($CR$):** $CR = \frac{CI}{RI_n}$, donde $RI_4 = 0.90$ (según tabla de Saaty). Si $CR < 0.10$, los juicios son consistentes.

In [3]:
# Obtenemos la matriz agregada del widget (o generada por ti)
m_criterios = widget_criterios.matriz_agregada

# Ejecutamos el desglose matemático transparente
res_criterios = md.calcular_ahp_paso_a_paso(m_criterios, md.PALMOR_CRITERIOS)

print("=== TABLA DIDÁCTICA DE CONSISTENCIA Y PESOS (CRITERIOS) ===")
display(res_criterios['tabla_resumen'].style.format({
    "Peso (w_i)": "{:.4f}",
    "% Prioridad": "{:.2f}%",
    "(A · w)_i": "{:.4f}",
    "Ratio (A·w)/w": "{:.4f}"
}))

print(f"\nResultados de consistencia:")
print(f"  λ_max = {res_criterios['lambda_max']:.4f}")
print(f"  CI    = {res_criterios['CI']:.4f}")
print(f"  RI    = {res_criterios['RI']:.2f} (para n={res_criterios['n']})")
print(f"  CR    = {res_criterios['CR']:.4f} -> {'✅ Consistente (CR < 0.10)' if res_criterios['es_consistente'] else '⚠️ Inconsistente (CR >= 0.10)'}")

=== TABLA DIDÁCTICA DE CONSISTENCIA Y PESOS (CRITERIOS) ===


,Peso (w_i),% Prioridad,(A · w)_i,Ratio (A·w)/w
Alcance,0.1604,16.04%,0.6433,4.0105
Autonomia,0.2502,25.02%,1.0019,4.0051
Infraestructura,0.4875,48.75%,1.9566,4.0131
Madurez,0.1019,10.19%,0.4083,4.0062



Resultados de consistencia:
  λ_max = 4.0087
  CI    = 0.0029
  RI    = 0.90 (para n=4)
  CR    = 0.0032 -> ✅ Consistente (CR < 0.10)


In [4]:
# Visualización didáctica de los pesos de los criterios
fig_criterios = md.graficar_pesos(
    nombres=md.PALMOR_CRITERIOS,
    pesos=res_criterios['pesos'],
    titulo="Ponderación Relativa de Criterios (Palmor IoT)"
)
plt.show()

## Paso 2: Comparación de Tecnologías bajo cada Criterio

Ahora comparamos las 4 alternativas candidatas (**LoRaWAN, GSM/GPRS, Sigfox, Zigbee**) de manera independiente bajo cada uno de los 4 criterios.

* En el siguiente widget interactivo puedes **cambiar de criterio** usando el menú desplegable.
* Puedes hacer clic en **'Cargar Palmor (4 Criterios)'** para cargar de inmediato los juicios técnicos del caso de clase.

In [5]:
# Widget interactivo con selector desplegable de criterio
widget_alternativas = md.AHPAlternativasWidget(
    alternativas=md.PALMOR_TECNOLOGIAS,
    criterios=md.PALMOR_CRITERIOS,
    n_expertos=3,
    matrices_iniciales=md.PALMOR_MATRICES_ALTERNATIVAS
)
widget_alternativas.mostrar()

In [6]:
# Matriz de Prioridades Locales: cada columna es el vector de pesos (w) bajo un criterio
W_locales = widget_alternativas.prioridades_locales

df_locales = pd.DataFrame(
    W_locales,
    index=md.PALMOR_TECNOLOGIAS,
    columns=md.PALMOR_CRITERIOS
)

print("=== MATRIZ DE PRIORIDADES LOCALES [Alternativas x Criterios] ===")
display(df_locales.style.format("{:.4f}").background_gradient(cmap='Blues', axis=0))

=== MATRIZ DE PRIORIDADES LOCALES [Alternativas x Criterios] ===


,Alcance,Autonomia,Infraestructura,Madurez
LoRaWAN,0.2368,0.5624,0.0921,0.3867
GSM/GPRS,0.1996,0.0698,0.2692,0.1553
Sigfox,0.5017,0.1904,0.5453,0.0713
Zigbee,0.0619,0.1774,0.0933,0.3867


## Paso 3: Síntesis Global y Ranking Final

La prioridad global de cada tecnología se obtiene mediante la suma ponderada de sus prioridades locales multiplicadas por el peso de cada criterio:

$$w_{global} = W_{locales} \cdot w_{criterios}$$

$$\begin{bmatrix} P(LoRaWAN) \\ P(GSM) \\ P(Sigfox) \\ P(Zigbee) \end{bmatrix} = \begin{bmatrix} w_{11} & w_{12} & w_{13} & w_{14} \\ w_{21} & w_{22} & w_{23} & w_{24} \\ w_{31} & w_{32} & w_{33} & w_{34} \\ w_{41} & w_{42} & w_{43} & w_{44} \end{bmatrix} \begin{bmatrix} w_{Alcance} \\ w_{Autonomia} \\ w_{Infraestructura} \\ w_{Madurez} \end{bmatrix}$$

In [7]:
# Multiplicación matricial (Síntesis)
prioridad_global = md.sintetizar_ahp(W_locales, res_criterios['pesos'])

# Tabla de Ranking ordenado
ranking = sorted(zip(md.PALMOR_TECNOLOGIAS, prioridad_global), key=lambda x: -x[1])
df_ranking = pd.DataFrame([
    {"Puesto": f"{i+1}º", "Tecnología": t, "Prioridad Global": p, "%": f"{p*100:.2f}%"}
    for i, (t, p) in enumerate(ranking)
])

print("=== RANKING FINAL AHP ===")
display(df_ranking)

=== RANKING FINAL AHP ===


,Puesto,Tecnología,Prioridad Global,%
0,1º,Sigfox,0.401248,40.12%
1,2º,LoRaWAN,0.262997,26.30%
2,3º,GSM/GPRS,0.196552,19.66%
3,4º,Zigbee,0.139204,13.92%


In [8]:
# Gráfico didáctico de barras apiladas: Contribución de cada criterio a la decisión
fig_sintesis = md.graficar_sintesis_apilada(
    alternativas=md.PALMOR_TECNOLOGIAS,
    criterios=md.PALMOR_CRITERIOS,
    prioridades_locales=W_locales,
    pesos_criterios=res_criterios['pesos'],
    prioridad_global=prioridad_global
)
plt.show()

findfont: Failed to find font weight medium, now using 400.


## Paso 4: Validación Cruzada contra `pyDecision`

Para certificar que nuestro cálculo paso a paso en NumPy es 100% riguroso, lo contrastamos contra la función `ahp_method` de la librería `pyDecision`.

In [9]:
# Verificación contra pyDecision
w_py_criterios, cr_py_criterios = ahp_method(m_criterios, wd='m')

print("Resultados de Validación:")
print(f"  Pesos NumPy:      {np.round(res_criterios['pesos'], 4)}")
print(f"  Pesos pyDecision: {np.round(w_py_criterios, 4)}")
print(f"  CR NumPy:         {res_criterios['CR']:.4f}")
print(f"  CR pyDecision:    {cr_py_criterios:.4f}")

assert np.allclose(res_criterios['pesos'], w_py_criterios, atol=1e-5), "¡Discrepancia en pesos!"
assert np.isclose(res_criterios['CR'], cr_py_criterios, atol=1e-4), "¡Discrepancia en CR!"
print("\n🎉 ¡Validación exitosa! El desarrollo paso a paso coincide al 100% con pyDecision.")

Resultados de Validación:
  Pesos NumPy:      [0.1604 0.2502 0.4875 0.1019]
  Pesos pyDecision: [0.1604 0.2502 0.4875 0.1019]
  CR NumPy:         0.0032
  CR pyDecision:    0.0032

🎉 ¡Validación exitosa! El desarrollo paso a paso coincide al 100% con pyDecision.


---

### 📝 Anexo Didáctico: ¿Cómo ingresar juicios en Python Puro sin Widgets?
Si prefieres no usar widgets interactivos (por ejemplo en un script automatizado), puedes usar la función `md.construir_matriz` con un diccionario de pares únicos. Nunca necesitas escribir matrices 2D con fracciones invertidas a mano:

```python
# Ejemplo: definir solo los pares que conoces
juicios = {
    ("Alcance", "Autonomia"): 2,          # Alcance es 2 veces más importante
    ("Alcance", "Infraestructura"): 1/3,  # Infraestructura es 3 veces más importante
    ("Alcance", "Madurez"): 2,
    ("Autonomia", "Infraestructura"): 1/4,
    ("Autonomia", "Madurez"): 1,
    ("Infraestructura", "Madurez"): 4,
}

# La función completa automáticamente la diagonal en 1 y los inversos recíprocos:
m_ejemplo = md.construir_matriz(md.PALMOR_CRITERIOS, juicios)
```